# Event Response Analysis: Prediction Markets vs. Polls

**DS2500 — Programming with Data | Northeastern University**  
**Team:** Sebastian Brookes, Shiven Mishra

How did prediction markets (Polymarket) and traditional polls (FiveThirtyEight) respond to
major campaign events during the 2024 U.S. presidential election? This notebook measures
**reaction speed**, **intensity**, and **directional accuracy** for five key events.


In [ ]:
import sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# add project root to path so we can import from src/
PROJECT_ROOT = str(Path.cwd().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.analysis.events.event_response import (
    EVENTS,
    load_data,
    compute_swing_average,
    compute_538_swing_timeseries,
    build_reaction_summary,
    compute_raw_indexed_window,
)
from src.visualize.event_plots import (
    Theme,
    apply_style,
    format_header,
    TIMELINE_START,
    TIMELINE_END,
    EVENT_LABEL_LEVELS,
    SOURCES,
)

THEME = Theme()
apply_style()

# lower DPI for inline display
plt.rcParams["figure.dpi"] = 150

## Methodology

### Events analyzed

We focus on **five major campaign events** that fell within the March–September 12, 2024
overlap window (the period where both Polymarket and FiveThirtyEight have daily data for
swing states):

| #   | Event                       | Date    | Expected Direction |
| --- | --------------------------- | ------- | ------------------ |
| 1   | Biden-Trump Debate          | June 27 | pro-Trump          |
| 2   | Trump Assassination Attempt | July 13 | pro-Trump          |
| 3   | Biden Drops Out             | July 21 | pro-Harris         |
| 4   | Walz VP Pick                | Aug 6   | neutral            |
| 5   | Harris-Trump Debate         | Sept 10 | pro-Harris         |

### Why swing states?

National polls and probabilities wash out the signal: California will go blue and Alabama
will go red regardless of campaign events. By averaging across the seven swing states
(AZ, GA, MI, NV, NC, PA, WI), we isolate the _marginal_ effect of each event on the
states that actually decide the election.

### Measuring reactions

For each event, we compute:

- **Baseline** — average Trump lead over the 3 days before the event
- **Immediate** — average Trump lead on the event day and the day after
- **Settled** — average Trump lead over days 2–7 after the event
- **Shift** — settled minus baseline (the sustained change)
- **z-score** — shift divided by the source’s daily volatility (standard deviation of
  day-over-day changes), so we can compare across sources on a common scale


In [ ]:
pm, p538 = load_data()

print(f"Polymarket:      {pm.shape[0]:,} rows, {pm.shape[1]} cols")
print(f"FiveThirtyEight: {p538.shape[0]:,} rows, {p538.shape[1]} cols")
print()
print("Polymarket sample:")
display(pm.head())
print("\nFiveThirtyEight sample:")
display(p538.head())

In [ ]:
pm_swing = compute_swing_average(pm)
p538_swing = compute_538_swing_timeseries(p538)

print(f"Polymarket swing avg:      {pm_swing.index.min().date()} to {pm_swing.index.max().date()}  ({len(pm_swing)} days)")
print(f"FiveThirtyEight swing avg: {p538_swing.index.min().date()} to {p538_swing.index.max().date()}  ({len(p538_swing)} days)")

In [ ]:
# --- Event Timeline: dual-axis swing-state averages ---

def _filter_window(series, start, end):
    return series[(series.index >= start) & (series.index <= end)]

def _align_dual_axes(ax_left, ax_right):
    """Rescale both y-axes so their zero lines sit at the same height."""
    l_min, l_max = ax_left.get_ylim()
    r_min, r_max = ax_right.get_ylim()
    if l_max - l_min == 0 or r_max - r_min == 0:
        return
    l_ratio = abs(l_min) / (l_max - l_min)
    r_ratio = abs(r_min) / (r_max - r_min)
    ratio = max(l_ratio, r_ratio)
    l_span = l_max if l_max > 0 else abs(l_min)
    r_span = r_max if r_max > 0 else abs(r_min)
    if ratio >= 1:
        ax_left.set_ylim(-l_span, l_span)
        ax_right.set_ylim(-r_span, r_span)
        return
    scale = ratio / (1 - ratio)
    ax_left.set_ylim(-scale * l_span, l_span)
    ax_right.set_ylim(-scale * r_span, r_span)

pm_filtered = _filter_window(pm_swing, TIMELINE_START, TIMELINE_END)
p538_filtered = _filter_window(p538_swing, TIMELINE_START, TIMELINE_END)

fig, ax1 = plt.subplots(figsize=(14, 6.5))
fig.subplots_adjust(top=0.78, right=0.82)

ax2 = ax1.twinx()

line1 = ax1.plot(
    pm_filtered.index, pm_filtered.values,
    color=THEME.PM_COLOR, linewidth=2.5, zorder=4,
    label="Polymarket (Win Prob.)",
)
line2 = ax2.plot(
    p538_filtered.index, p538_filtered.values,
    color=THEME.F38_COLOR, linewidth=2.2, zorder=3,
    label="FiveThirtyEight (Vote Share %)",
)
ax2.spines["right"].set_visible(False)

ax1.tick_params(axis="y", colors=THEME.PM_COLOR, labelsize=11)
ax1.set_ylabel("Win Probability", color=THEME.PM_COLOR,
               fontweight="bold", fontsize=11, labelpad=10)
ax2.tick_params(axis="y", colors=THEME.F38_COLOR, labelsize=11)
ax2.set_ylabel("Vote Share %", color=THEME.F38_COLOR,
               fontweight="bold", fontsize=11, rotation=-90, labelpad=20)

_align_dual_axes(ax1, ax2)
ax1.axhline(0, color=THEME.TEXT_MAIN, linewidth=1.2, zorder=2)

lines = line1 + line2
labels = [l.get_label() for l in lines]
leg = ax1.legend(
    lines, labels,
    loc="upper right", bbox_to_anchor=(1.0, 1.32),
    ncol=1, frameon=False, fontsize=11,
    handletextpad=0.5, labelspacing=0.6,
)
for text, color in zip(leg.get_texts(), [THEME.PM_COLOR, THEME.F38_COLOR]):
    text.set_color(color)
    text.set_fontweight("bold")

start_date = pm_filtered.index.min() if not pm_filtered.empty else TIMELINE_START
ax1.annotate("Trump leads", xy=(start_date, 0), xytext=(10, 8),
             textcoords="offset points", color=THEME.TEXT_MUTED, fontsize=9, va="bottom")
ax1.annotate("Biden / Harris leads", xy=(start_date, 0), xytext=(10, -8),
             textcoords="offset points", color=THEME.TEXT_MUTED, fontsize=9, va="top")

# event bands and labels
_, y_max = ax1.get_ylim()
for event in EVENTS:
    event_date = pd.Timestamp(event["date"])
    if event_date < TIMELINE_START or event_date > TIMELINE_END:
        continue
    ax1.axvspan(
        event_date - pd.Timedelta(days=1),
        event_date + pd.Timedelta(days=1),
        color=THEME.EVENT_BAND, zorder=1,
    )
    level = EVENT_LABEL_LEVELS.get(event["name"], 0)
    ax1.annotate(
        event["name"], xy=(event_date, y_max),
        xytext=(0, 6 + 14 * level), textcoords="offset points",
        ha="center", va="bottom", fontsize=8.5,
        color=THEME.TEXT_MAIN, annotation_clip=False,
    )

ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b."))

format_header(
    fig,
    "How Prediction Markets and Polls Tracked the Campaign",
    "Swing-state average Trump lead, showing Polymarket probability vs. FiveThirtyEight vote share.",
)
plt.show()

### Timeline interpretation

Both sources follow the same broad narrative arc: Trump builds a lead through June and
early July (debate performance + assassination attempt sympathy), then **Biden dropping
out on July 21 is the single biggest inflection point of the campaign** — both series
reverse sharply as Harris replaces Biden.

Key observations:

- **Polymarket reacted instantly** to the dropout — traders repriced overnight. The
  FiveThirtyEight average took several days to reflect the shift as new polls trickled in.
- Both sources show a modest Harris bump from the September debate, though the signal is
  weaker than the dropout effect.
- The dual-axis alignment keeps the zero line common to both series so that "above zero =
  Trump leads" and "below zero = Harris leads" is visually consistent.


In [ ]:
summary = build_reaction_summary(pm, p538)

display_cols = ["event", "source", "expected", "baseline", "settled",
                "shift", "shift_z", "direction_match", "has_data"]
display(summary[display_cols].style.format({
    "baseline": "{:.4f}",
    "settled": "{:.4f}",
    "shift": "{:+.4f}",
    "shift_z": "{:+.1f}σ",
}).set_caption("Reaction summary: all events × both sources"))

In [ ]:
# --- Reaction Scoreboard: z-scored shift intensity per event × source ---

event_names = [event["name"] for event in EVENTS]
y_pos = np.arange(len(event_names))
bar_h = 0.35

values = (
    summary[["event", "source", "shift_z", "direction_match"]]
    .drop_duplicates(subset=["event", "source"], keep="last")
    .set_index(["event", "source"])
)

fig, ax = plt.subplots(figsize=(10, 6.5))
fig.subplots_adjust(top=0.78, left=0.25)
ax.axvline(0, color=THEME.TEXT_MAIN, linewidth=1, zorder=2)

max_abs_z = 0.0
for i, event_name in enumerate(event_names):
    for source, _, color, y_offset in SOURCES:
        key = (event_name, source)
        if key not in values.index:
            continue
        row = values.loc[key]
        z_score = row["shift_z"]
        if pd.isna(z_score):
            continue

        is_correct = row["direction_match"] == "correct"
        face_color = color if is_correct else "white"
        alpha = 0.95 if is_correct else 0.65

        ax.barh(
            y_pos[i] + y_offset, z_score,
            height=bar_h * 0.86, facecolor=face_color,
            edgecolor=color, linewidth=1.4, alpha=alpha, zorder=3,
        )
        label_pad = 0.08 if z_score >= 0 else -0.08
        ax.text(
            z_score + label_pad, y_pos[i] + y_offset,
            f"{z_score:+.1f}σ", ha="left" if z_score >= 0 else "right",
            va="center", fontsize=8.5, color=color, fontweight="bold",
        )
        max_abs_z = max(max_abs_z, abs(z_score))

max_abs_z = max(max_abs_z, 1.0)
x_pad = max(0.25, 0.15 * max_abs_z)
ax.set_xlim(-max_abs_z - x_pad, max_abs_z + x_pad)
ax.set_yticks(y_pos)
ax.set_yticklabels(event_names)
ax.invert_yaxis()
ax.set_xlabel("Reaction intensity (z-score, normalized to daily volatility)")
ax.spines["bottom"].set_visible(False)
ax.tick_params(axis="x", which="both", bottom=False, labelbottom=False)
ax.grid(False)

legend_elements = [
    Patch(facecolor=THEME.TEXT_MAIN, edgecolor=THEME.TEXT_MAIN, label="Expected direction"),
    Patch(facecolor="white", edgecolor=THEME.TEXT_MAIN, linewidth=1.4, label="Unexpected direction"),
    Line2D([0], [0], color=THEME.PM_COLOR, lw=4, label="Polymarket"),
    Line2D([0], [0], color=THEME.F38_COLOR, lw=4, label="FiveThirtyEight"),
]
ax.legend(
    handles=legend_elements, loc="lower right",
    bbox_to_anchor=(1.0, 1.02), ncol=2, fontsize=9, frameon=False,
)

format_header(
    fig,
    "Event Reaction Scoreboard",
    "Which source reacted more intensely, and did they move in the expected direction?",
)
plt.show()

### Scoreboard interpretation

The scoreboard reveals two important patterns:

1. **Direction accuracy** — Polymarket shifted in the expected direction for 4 of 5 events;
   FiveThirtyEight managed 3 of 5. Both sources showed an unexpected pro-Trump reaction to
   the Walz VP pick (labelled neutral/expected, so any large move counts as overreaction).

2. **Intensity** — Polymarket’s z-scored reactions are consistently larger, meaning that
   markets moved further relative to their own normal daily noise. This is consistent with
   markets processing new information in near-real-time, while poll aggregates smooth and
   dilute the signal over days.

3. **Data gaps** — FiveThirtyEight has a coverage gap around the Harris-Trump debate
   (Sept 10), which falls near the end of our overlap window (Sept 12). This limits our
   ability to measure the settled reaction for that event in the polling data.


In [ ]:
# --- Biden Dropout Deep Dive: indexed event study ---

def _steepest_drop_day(series):
    diffs = series.diff().dropna()
    if diffs.empty:
        return None
    return diffs.idxmin()

def _map_y_between_axes(value, src_ax, dst_ax):
    src_min, src_max = src_ax.get_ylim()
    dst_min, dst_max = dst_ax.get_ylim()
    if src_max == src_min:
        return dst_min
    return dst_min + ((value - src_min) / (src_max - src_min)) * (dst_max - dst_min)

hero_date = next(e["date"] for e in EVENTS if e["name"] == "Biden Drops Out")
pm_window = compute_raw_indexed_window(pm_swing, hero_date, pre_days=1, scale=100)
p538_window = compute_raw_indexed_window(p538_swing, hero_date, pre_days=1, scale=1)

fig, ax1 = plt.subplots(figsize=(10, 6.5))
fig.subplots_adjust(top=0.82, right=0.95)
ax1.grid(axis="y", alpha=0.5)

ax2 = ax1.twinx()
ax1.step(pm_window.index, pm_window.values, where="post",
         color=THEME.PM_COLOR, linewidth=2.5)
ax2.plot(p538_window.index, p538_window.values,
         color=THEME.F38_COLOR, linewidth=2.3)
ax2.spines["right"].set_visible(False)

_align_dual_axes(ax1, ax2)
ax1.tick_params(axis="y", colors=THEME.PM_COLOR)
ax2.tick_params(axis="y", colors=THEME.F38_COLOR)
ax1.axvline(0, color=THEME.TEXT_MAIN, linewidth=1, linestyle=":", zorder=2)
ax1.axhline(0, color=THEME.TEXT_MAIN, linewidth=1, zorder=2)
ax1.text(
    0, 0.04, "Biden withdraws",
    color=THEME.TEXT_MAIN, fontsize=9, ha="center", va="bottom",
    transform=ax1.get_xaxis_transform(),
    bbox=dict(facecolor="white", edgecolor="none", pad=4), zorder=3,
)

ax1.set_ylabel("Polymarket\n(percentage points)",
               color=THEME.PM_COLOR, fontsize=11, fontweight="bold",
               linespacing=1.4, labelpad=10)
ax2.set_ylabel("FiveThirtyEight\n(vote share points)",
               color=THEME.F38_COLOR, fontsize=11, fontweight="bold",
               linespacing=1.4, rotation=270, labelpad=40)

# lag annotation
pm_lag = _steepest_drop_day(pm_window)
p538_lag = _steepest_drop_day(p538_window)
if pm_lag is not None and p538_lag is not None and p538_lag > pm_lag:
    pm_floor = float(pm_window.min()) if not pm_window.empty else 0.0
    y_bracket = 0.35 * pm_floor if pm_floor < 0 else -0.1
    ax1.annotate(
        "", xy=(p538_lag, y_bracket), xytext=(pm_lag, y_bracket),
        arrowprops=dict(arrowstyle="|-|,widthA=0.2,widthB=0.2",
                        color=THEME.TEXT_MAIN, lw=1.2),
    )
    ax1.annotate(
        f"{int(p538_lag - pm_lag)}-day lag",
        xy=((pm_lag + p538_lag) / 2, y_bracket),
        xytext=(0, 5), textcoords="offset points",
        ha="center", va="bottom", fontsize=10,
        fontweight="bold", color=THEME.TEXT_MAIN,
    )

    pm_value = pm_window.loc[pm_lag]
    ax1.plot([pm_lag, pm_lag], [y_bracket, pm_value],
             color=THEME.PM_COLOR, linestyle=":", lw=1.2, alpha=0.7)
    ax1.plot([pm_lag], [pm_value], marker="o",
             color=THEME.PM_COLOR, markersize=5)

    p538_value = p538_window.loc[p538_lag]
    y_bracket_ax2 = _map_y_between_axes(y_bracket, ax1, ax2)
    ax2.plot([p538_lag, p538_lag], [y_bracket_ax2, p538_value],
             color=THEME.F38_COLOR, linestyle=":", lw=1.2, alpha=0.7)
    ax2.plot([p538_lag], [p538_value], marker="o",
             color=THEME.F38_COLOR, markersize=5)

ax1.set_xlabel("Days from event")

format_header(
    fig,
    "Market Traders Spotted the Harris Surge Before Polling Caught Up",
    "Change in probability and vote share, indexed to the day before Biden dropped out.",
)
plt.show()

### Biden dropout interpretation

The indexed event study makes the speed difference vivid:

- **Polymarket** began repricing on Day 0 (July 21), but the steepest single-day drops
  came on **Days 1 and 3** — not Day 0 itself. Day 0 was a Sunday afternoon announcement
  with thin trading volume; Day 1 (Monday) brought a full trading session plus Harris's
  rapid consolidation of delegate endorsements, giving traders concrete information to
  price in. The Day 3 drop coincides with the wave of major endorsements (Pelosi, Schumer,
  and the Congressional Black Caucus all backed Harris within 72 hours), which removed the
  last remaining uncertainty about a contested convention.
- **FiveThirtyEight** took several more days to reflect the shift. The lag annotation on
  the chart shows the gap between each source's steepest single-day drop.

**Why the difference?** Polymarket trades 24/7 — traders can reprice the moment news
breaks. But even markets don't fully reprice on a single day: the dropout itself was only
the first domino, and each subsequent endorsement cascade narrowed the remaining
uncertainty. FiveThirtyEight aggregates polls, and polls take time to field, collect
responses, weight, and publish. This structural latency means poll aggregates always lag
the true shift in public sentiment by at least a few days — and events that unfold over
multiple news cycles (like a contested-then-settled nomination) compound the delay.


## Conclusions

### Summary of findings

1. **Markets respond faster.** Polymarket repriced swing-state probabilities within hours
   of major events, while FiveThirtyEight's poll aggregate lagged by days.

2. **Markets are more often directionally correct.** Polymarket shifted in the expected
   direction for 4 of 5 events; FiveThirtyEight matched expectations for 3 of 5.

3. **Markets produce stronger signals.** Z-scored reaction magnitudes are consistently
   larger for Polymarket, reflecting a higher signal-to-noise ratio relative to each
   source's own daily volatility.

### Limitations

- **Small event sample** — only 5 events, limiting statistical power for direction accuracy.
- **Overlap window ends Sept 12** — we miss the final two months of the campaign where late
  polls converge toward election day.
- **538 data gaps** — irregular poll publication creates coverage gaps, especially near the
  end of our window (Harris-Trump debate).
- **Apples-to-oranges scale** — Polymarket reports win probabilities while 538 reports vote
  shares. Z-scoring normalizes this, but the underlying quantities are fundamentally
  different.

### So what?

The practical takeaway is that markets and polls answer different questions on different
timescales. A newsroom building a live election dashboard should **weight market data for
breaking-event coverage** (assassination attempts, candidate withdrawals, debate nights)
and **switch to poll data for weekly or monthly trend reports** (demographic shifts, turnout
modeling, regional swings). Neither source alone tells the full story — but using both
without understanding their structural lag differences will produce misleading analysis.

For campaigns themselves, the implication is sharper: if your internal tracking relies
solely on poll aggregates, you are seeing the world on a 3–5 day delay during the moments
that matter most. Market prices won't tell you _why_ the race is shifting, but they will
tell you _that_ it is shifting — days before your pollster can confirm it.
